In [31]:
# Import required libraries

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns if needed
pd.set_option('display.max_columns', None)

# Make plots look cleaner
plt.style.use('ggplot')


In [32]:
stocks = [
    "HDFCBANK.NS",    # Banking
    "INFY.NS",        # IT
    "RELIANCE.NS",    # Energy
    "HINDUNILVR.NS"   # FMCG
]

benchmark = "^NSEI"

In [33]:
start_date = "2019-01-01"
end_date = "2026-01-01"

In [34]:
benchmark_prices=yf.download(benchmark,
                             start='2019-01-01',
                             end='2026-01-01',
                             auto_adjust=True,
                             progress=True)['Close']
               

[*********************100%***********************]  1 of 1 completed


In [35]:
tickers = [
    "HDFCBANK.NS",
    "INFY.NS",
    "RELIANCE.NS",
    "HINDUNILVR.NS"
]

prices = pd.DataFrame()

for ticker in tickers:
    
    data = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False
    )
    
    # Extract Adjusted Close
    prices[ticker] = data["Adj Close"].squeeze()

prices.head()

,HDFCBANK.NS,INFY.NS,RELIANCE.NS,HINDUNILVR.NS
Date,,,,
2019-01-01,496.185852,541.725769,496.196411,1591.314453
2019-01-02,491.658478,544.984070,489.733887,1576.735962
2019-01-03,487.801056,545.065674,483.691864,1580.093628
2019-01-04,489.106049,538.467468,486.303467,1574.174072
2019-01-07,489.845215,547.142761,489.003540,1577.045532


In [36]:
# ============================================================
# Rename Columns
# ============================================================

prices.columns = [
    "HDFC Bank",
    "Infosys",
    "Reliance",
    "HUL"
]

In [37]:
# ============================================================
# Display First Five Observations
# ============================================================

prices.head()

,HDFC Bank,Infosys,Reliance,HUL
Date,,,,
2019-01-01,496.185852,541.725769,496.196411,1591.314453
2019-01-02,491.658478,544.984070,489.733887,1576.735962
2019-01-03,487.801056,545.065674,483.691864,1580.093628
2019-01-04,489.106049,538.467468,486.303467,1574.174072
2019-01-07,489.845215,547.142761,489.003540,1577.045532


In [38]:
# Dataset Information

prices.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1730 entries, 2019-01-01 to 2025-12-31
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   HDFC Bank  1730 non-null   float64
 1   Infosys    1730 non-null   float64
 2   Reliance   1730 non-null   float64
 3   HUL        1730 non-null   float64
dtypes: float64(4)
memory usage: 67.6 KB


In [39]:
# Summary Statistics

prices.describe().round(2)


,HDFC Bank,Infosys,Reliance,HUL
count,1730.00,1730.00,1730.00,1730.00
mean,704.62,1217.08,1064.15,2171.68
std,140.05,382.47,294.89,303.83
min,357.66,442.78,393.66,1465.85
25%,592.58,895.54,885.19,1965.67
50%,708.81,1329.37,1116.88,2231.22
75%,788.85,1491.22,1261.37,2404.42
max,996.42,1901.08,1581.82,2909.86


### Data Cleaning

Before conducting financial analysis, the dataset is checked for missing values.

Missing observations can occur due to trading holidays or incomplete downloads.

In [40]:
# ============================================================
# Check Missing Values
# ============================================================

prices.isnull().sum()

HDFC Bank    0
Infosys      0
Reliance     0
HUL          0
dtype: int64

In [41]:
# Total Missing Values

prices.isnull().sum().sum()

np.int64(0)

In [42]:
# ============================================================
# Check Dataset Dimensions
# ============================================================

print("Rows and Columns:", prices.shape)

print("\n")

print(prices.head())

Rows and Columns: (1730, 4)


             HDFC Bank     Infosys    Reliance          HUL
Date                                                       
2019-01-01  496.185852  541.725769  496.196411  1591.314453
2019-01-02  491.658478  544.984070  489.733887  1576.735962
2019-01-03  487.801056  545.065674  483.691864  1580.093628
2019-01-04  489.106049  538.467468  486.303467  1574.174072
2019-01-07  489.845215  547.142761  489.003540  1577.045532


In [43]:
# ============================================================
# Dataset Summary
# ============================================================

summary = pd.DataFrame({

    "Data Type": prices.dtypes,
    "Missing Values": prices.isnull().sum(),
    "Minimum Adjusted Close": prices.min(),
    "Maximum Adjusted Close": prices.max(),
    "Average Adjusted Close": prices.mean()

})

summary.round(2)

,Data Type,Missing Values,Minimum Adjusted Close,Maximum Adjusted Close,Average Adjusted Close
HDFC Bank,float64,0,357.66,996.42,704.62
Infosys,float64,0,442.78,1901.08,1217.08
Reliance,float64,0,393.66,1581.82,1064.15
HUL,float64,0,1465.85,2909.86,2171.68


In [44]:
#returns over entire period under study
returns = (
prices.iloc[-1] /
prices.iloc[0] - 1
) * 100

returns = returns.round(2)

returns

HDFC Bank     96.51
Infosys      191.88
Reliance     215.03
HUL           44.07
dtype: float64

In [45]:
# Combine stock and benchmark closing prices by date
all_prices = pd.concat(
    [
        prices,
        benchmark_prices
    ],
    axis=1,
    join="inner"
).dropna()

print(all_prices.head())
print(all_prices.isna().sum())
print(all_prices.shape)

             HDFC Bank     Infosys    Reliance          HUL         ^NSEI
Date                                                                     
2019-01-02  491.658478  544.984070  489.733887  1576.735962  10792.500000
2019-01-03  487.801056  545.065674  483.691864  1580.093628  10672.250000
2019-01-04  489.106049  538.467468  486.303467  1574.174072  10727.349609
2019-01-07  489.845215  547.142761  489.003540  1577.045532  10771.799805
2019-01-08  485.768280  545.798706  488.959290  1564.587769  10802.150391
HDFC Bank    0
Infosys      0
Reliance     0
HUL          0
^NSEI        0
dtype: int64
(1727, 5)


In [46]:
all_prices.rename(columns={'^NSEI':'Nifty 50'},inplace=True)

In [47]:
all_prices

,HDFC Bank,Infosys,Reliance,HUL,Nifty 50
Date,,,,,
2019-01-02,491.658478,544.984070,489.733887,1576.735962,10792.500000
2019-01-03,487.801056,545.065674,483.691864,1580.093628,10672.250000
2019-01-04,489.106049,538.467468,486.303467,1574.174072,10727.349609
2019-01-07,489.845215,547.142761,489.003540,1577.045532,10771.799805
2019-01-08,485.768280,545.798706,488.959290,1564.587769,10802.150391
...,...,...,...,...,...
2025-12-24,980.975220,1628.167480,1551.028687,2259.220215,26142.099609
2025-12-26,975.958191,1621.022095,1552.024048,2262.387939,26042.300781
2025-12-29,975.564697,1609.863525,1538.486694,2270.208496,25942.099609


In [24]:
daily_returns = all_prices.drop(columns="Nifty 50").pct_change().dropna()

nifty_returns = all_prices["Nifty 50"].pct_change().dropna()


In [52]:
# ============================================================
# Save Clean Data
# ============================================================

from pathlib import Path

DOWNLOADS_DIR = Path.home() / "Downloads"

prices.to_csv(
    DOWNLOADS_DIR / "stock_prices.csv",
    date_format="%Y-%m-%d"
)

benchmark_prices.to_csv(
    DOWNLOADS_DIR / "nifty50.csv",
    date_format="%Y-%m-%d"
)

In [50]:
prices=all_prices.drop('Nifty 50',axis=1)
prices.head()

,HDFC Bank,Infosys,Reliance,HUL
Date,,,,
2019-01-02,491.658478,544.984070,489.733887,1576.735962
2019-01-03,487.801056,545.065674,483.691864,1580.093628
2019-01-04,489.106049,538.467468,486.303467,1574.174072
2019-01-07,489.845215,547.142761,489.003540,1577.045532
2019-01-08,485.768280,545.798706,488.959290,1564.587769
